# Cross-Test: PortPy vs QuantStats vs Empyrical

This notebook tests `portpy` metric calculations against the equivalent functions in `quantstats` and `empyrical` using live data fetched via the Alpaca API.

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from dotenv import load_dotenv

# External metric libraries
import quantstats as qs
import empyrical as ep

# Alpaca
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

# PortPy
from portpy import Portfolio
import portpy.metrics as pm

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda v: f"{v:,.4f}")

## 1. Authentication & Setup

Load Alpaca API keys.

In [2]:
load_dotenv()

api_key = os.environ.get('ALPACA_KEY')
api_secret = os.environ.get('ALPACA_SECRET')

if not api_key or not api_secret:
    raise ValueError("API Keys not found in .env file! Please set ALPACA_KEY and ALPACA_SECRET")

client = StockHistoricalDataClient(api_key, api_secret)
print("Alpaca Client Initialized.")

Alpaca Client Initialized.


## 2. Fetch Data from Alpaca
We will fetch the last 3 years of daily bars for a standard portfolio (e.g., SPY, AAPL, AGG).

In [3]:
symbols = ["SPY", "AAPL", "AGG"]
start_date = pd.Timestamp.now() - pd.DateOffset(years=3)

request_params = StockBarsRequest(
    symbol_or_symbols=symbols,
    timeframe=TimeFrame.Day,
    start=start_date
)

bars = client.get_stock_bars(request_params)
df = bars.df

# Pivot to have symbols as columns and dates as index
prices = df.reset_index().pivot(index='timestamp', columns='symbol', values='close')
# STRIP TIMEZONE TO FIX QUANTSTATS BUG
prices.index = pd.to_datetime(prices.index).tz_localize(None).normalize()
prices = prices.dropna()
prices.head()

symbol,AAPL,AGG,SPY
timestamp,,,
2023-08-21,175.8400,95.1400,439.3400
2023-08-22,177.2300,95.2400,438.1500
2023-08-23,181.1200,96.1500,443.0300
2023-08-24,176.3800,95.9300,436.8900
2023-08-25,178.6100,95.8700,439.9700


## 3. Setup PortPy Portfolio

In [4]:
weights = {sym: 1.0/len(symbols) for sym in symbols}
portfolio = Portfolio(prices, weights=weights, risk_free_rate=0.0)

# Get the daily portfolio returns to feed into QuantStats and Empyrical
port_returns = portfolio.returns()

# Extract SPY as the benchmark for benchmark-relative metrics
benchmark_returns = prices['SPY'].pct_change().dropna()

# Align port_returns and benchmark_returns just in case
port_returns, benchmark_returns = port_returns.align(benchmark_returns, join='inner')
portfolio = Portfolio(prices.loc[port_returns.index], weights=weights, risk_free_rate=0.0)
port_returns = portfolio.returns()

print(f"Portfolio Total Return: {portfolio.metrics.total_return():.2%}")

Portfolio Total Return: 49.75%


## 4. Cross-Test Metrics

In [5]:
# Exhaustive comparison of all overlapping metrics
results = []

def add_res(metric_name, pp_val, qs_val=None, ep_val=None):
    results.append({"Metric": metric_name, "PortPy": pp_val, "QuantStats": qs_val, "Empyrical": ep_val})

# --- NON-BENCHMARK METRICS ---
add_res("CAGR", portfolio.metrics.cagr(), qs.stats.cagr(port_returns), ep.cagr(port_returns))
add_res("Volatility", portfolio.metrics.volatility(), qs.stats.volatility(port_returns, annualize=True), ep.annual_volatility(port_returns))
add_res("Sharpe Ratio", portfolio.metrics.sharpe_ratio(), qs.stats.sharpe(port_returns, rf=0.0), ep.sharpe_ratio(port_returns, risk_free=0.0))
add_res("Sortino Ratio", portfolio.metrics.sortino_ratio(), qs.stats.sortino(port_returns, rf=0.0), ep.sortino_ratio(port_returns, required_return=0.0))
add_res("Calmar Ratio", portfolio.metrics.calmar_ratio(), qs.stats.calmar(port_returns), ep.calmar_ratio(port_returns))
add_res("Max Drawdown", portfolio.metrics.max_drawdown(), qs.stats.max_drawdown(port_returns), ep.max_drawdown(port_returns))
add_res("Value at Risk (95%)", portfolio.metrics.value_at_risk(confidence=0.95), qs.stats.value_at_risk(port_returns), ep.value_at_risk(port_returns))
add_res("Conditional VaR / ES", portfolio.metrics.conditional_var(confidence=0.95), qs.stats.cvar(port_returns), ep.conditional_value_at_risk(port_returns))
add_res("Tail Ratio", portfolio.metrics.tail_ratio(), qs.stats.tail_ratio(port_returns), ep.tail_ratio(port_returns))
add_res("Skewness", portfolio.metrics.skewness(), qs.stats.skew(port_returns), None)
add_res("Kurtosis", portfolio.metrics.kurtosis(), qs.stats.kurtosis(port_returns), None)
add_res("Win Rate", portfolio.metrics.win_rate(), qs.stats.win_rate(port_returns), None)
add_res("Win/Loss Ratio", portfolio.metrics.win_loss_ratio(), qs.stats.win_loss_ratio(port_returns), None)
add_res("Gain to Pain Ratio", portfolio.metrics.gain_to_pain_ratio(), qs.stats.gain_to_pain_ratio(port_returns), None)
add_res("Ulcer Index", portfolio.metrics.ulcer_index(), qs.stats.ulcer_index(port_returns), None)
add_res("Omega Ratio", portfolio.metrics.omega_ratio(), qs.stats.omega(port_returns), ep.omega_ratio(port_returns, required_return=0.0))
add_res("Downside Deviation/Risk", portfolio.metrics.downside_deviation(), None, ep.downside_risk(port_returns))

# --- BENCHMARK-RELATIVE METRICS ---
add_res("Alpha", portfolio.metrics.alpha(benchmark=benchmark_returns), None, ep.alpha(port_returns, benchmark_returns))
add_res("Beta", portfolio.metrics.beta(benchmark=benchmark_returns), None, ep.beta(port_returns, benchmark_returns))
add_res("R-Squared", portfolio.metrics.r_squared(benchmark=benchmark_returns), qs.stats.r_squared(port_returns, benchmark_returns), None)
add_res("Information Ratio", portfolio.metrics.information_ratio(benchmark=benchmark_returns), qs.stats.information_ratio(port_returns, benchmark_returns), None)

# Treynor uses Beta in PortPy!
pp_beta = portfolio.metrics.beta(benchmark=benchmark_returns)
add_res("Treynor Ratio", portfolio.metrics.treynor_ratio(beta=pp_beta), qs.stats.treynor_ratio(port_returns, benchmark_returns), None)

add_res("Up Capture Ratio", portfolio.metrics.up_capture_ratio(benchmark=benchmark_returns), None, ep.up_capture(port_returns, benchmark_returns))
add_res("Down Capture Ratio", portfolio.metrics.down_capture_ratio(benchmark=benchmark_returns), None, ep.down_capture(port_returns, benchmark_returns))
add_res("Batting Average", portfolio.metrics.batting_average(benchmark=benchmark_returns), None, float(ep.batting_average(port_returns, benchmark_returns).iloc[0]))

df_compare = pd.DataFrame(results)

def color_diff(row):
    # Highlight differences > 1% (0.01)
    try:
        diff_qs = abs(row['PortPy'] - row['QuantStats']) if pd.notnull(row['QuantStats']) else 0
        diff_ep = abs(row['PortPy'] - row['Empyrical']) if pd.notnull(row['Empyrical']) else 0
        if diff_qs > 0.01 or diff_ep > 0.01:
            return ['background-color: #ffcccc'] * len(row)
    except:
        pass
    return [''] * len(row)

df_compare.style.apply(color_diff, axis=1).format({
    "PortPy": "{:.4f}",
    "QuantStats": "{:.4f}",
    "Empyrical": "{:.4f}"
}, na_rep="-")

,Metric,PortPy,QuantStats,Empyrical
0,CAGR,0.1455,0.1455,0.1455
1,Volatility,0.1324,0.1324,0.1324
2,Sharpe Ratio,1.0923,1.0923,1.0923
3,Sortino Ratio,1.6430,1.6430,1.6430
4,Calmar Ratio,0.8309,0.8309,0.8309
5,Max Drawdown,-0.1751,-0.1751,-0.1751
6,Value at Risk (95%),-0.0122,-0.0131,-0.0122
7,Conditional VaR / ES,-0.0181,-0.0198,-0.0181
8,Tail Ratio,1.0280,1.0280,1.0280
9,Skewness,0.9297,0.9297,-
